# Restore Vivienne's photos — free GPU (v2, robust)

Runs **CodeFormer** face-restoration on Google's free GPU and saves the result **to your Google Drive** (so it can't get lost if Colab disconnects).

**Steps:**
1. Top menu: **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**.
3. When it asks, **allow Google Drive access** (one click).
4. Wait ~15–25 min. When it's done you'll have **`restored.zip` in your Drive** (top level, "My Drive").
5. Tell me — I'll pull it from there, or you drop it in the repo.

Keep the tab open and the laptop awake while it runs. Originals are never touched.

In [ ]:
# 1) Confirm GPU (if this errors: Runtime → Change runtime type → T4 GPU, then Run all again)
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime → Change runtime type → T4 GPU, then Run all.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Save straight to your Google Drive (survives any disconnect). Click 'Allow' when prompted.
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive'
assert os.path.isdir(DRIVE), 'Drive did not mount — re-run this cell and click Allow.'
print('Drive mounted. Final zip will land at:', DRIVE + '/restored.zip')

In [ ]:
# 3) Install CodeFormer + dependencies, and PATCH the known torchvision/basicsr breakage
%cd /content
!rm -rf CodeFormer
!git clone -q https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -q -r requirements.txt
!python basicsr/setup.py develop 2>/dev/null
# The notorious fix: newer torchvision moved functional_tensor -> functional. Patch it everywhere.
!find / -name degradations.py -path '*basicsr*' 2>/dev/null -exec sed -i 's/functional_tensor/functional/g' {} +
# Pretrained weights
!python scripts/download_pretrained_models.py facelib
!python scripts/download_pretrained_models.py CodeFormer
print('install done')

In [ ]:
# 4) Pull the photos from the public repo
%cd /content
!rm -rf family && git clone -q --depth 1 https://github.com/MarcBittner/family.git
import os, shutil
shutil.rmtree('/content/inputs', ignore_errors=True); os.makedirs('/content/inputs')
src = '/content/family/montage/full'
n = 0
for f in sorted(os.listdir(src)):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy(os.path.join(src, f), '/content/inputs/' + f); n += 1
print('photos to restore:', n)

In [ ]:
# 5) Restore on the GPU (faces only = faster, fewer ways to break). ~15–25 min.
%cd /content/CodeFormer
!python inference_codeformer.py -w 0.7 --face_upsample -i /content/inputs -o /content/results
print('restoration finished')

In [ ]:
# 6) Rename (0001.jpg -> p0001.jpg), downsize for web, zip, and copy to Drive
%cd /content
import os, subprocess, shutil
fin = '/content/results/final_results'
out = '/content/restored'
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
cnt = 0
for f in sorted(os.listdir(fin)):
    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    stem = os.path.splitext(f)[0]
    subprocess.run(['convert', os.path.join(fin, f), '-resize', '1400x1400>',
                    '-quality', '88', os.path.join(out, 'p' + stem + '.jpg')])
    cnt += 1
print('restored & resized:', cnt)
!cd /content && rm -f restored.zip && zip -q -r restored.zip restored
shutil.copy('/content/restored.zip', '/content/drive/MyDrive/restored.zip')
mb = os.path.getsize('/content/restored.zip') // (1024*1024)
print(f'DONE — restored.zip ({mb} MB) is now in your Google Drive (My Drive).')

In [ ]:
# 7) (Optional backup) also try a direct browser download
try:
    from google.colab import files
    files.download('/content/restored.zip')
except Exception as e:
    print('Direct download skipped — your zip is safe in Google Drive. Error:', e)